# Secure Firmware Update

This exercise introduces practical examples of secure firmware update processes based on the tool ``swupdate``.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [ ]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "root" ssh root@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

## Exercise 9.1: Manual Firmware Update

In this exercise, you're going to update your device manually.

For this purpose, connect your device to this virtual machine and make sure it is reachable over the network.

### Current Boot Partition

Use the following commands to identify the current boot partition.

In [ ]:
execute_stm32_command('lsblk')

In [ ]:
execute_stm32_command('dmesg | grep "root="')

Which partition of your micro SD card is currently mounted as the root file system?

**Current Root File System Partition:**

*TODO*

### Update over a Web Interface

Open a new browser tab and visit ``192.168.13.13:8080``.

You will see the web interface of the ``swupdate`` service running on your device.

Click on *Messages* at the bootom of the web page to see some log messages of the later update process.

Now, open a file browser, take your downloaded ``.swu`` file, and move it to SWUpdate's web interface by drag-and-drop.

What happens during the update process? (according to the log messages)

**Important Steps of Update Process:**

*TODO*

### The New System

After the update process has finsihed and the device is up and running again, check the boot partition.

In [ ]:
execute_stm32_command('lsblk')

In [ ]:
execute_stm32_command('dmesg | grep "root="')

**Root File System Partition After Update:**

*TODO*

Inspect the ``.swu`` update file and answer the following questions:

* What file type is actually behind the ``.swu`` file?
* Which information is contained in the ``sw-description`` file?
* Which protection goals are archieved with the current construction?


**File Type Behind ``.swu`` Files:**

*TODO*

**Information in ``sw-description`` File:**

*TODO*

**Achieved Protection Goals for Update File:**

* Confidentiality: *YES/NO*
* Integrity: *YES/NO*
* Availability: *YES/NO*
* Authenticity: *YES/NO*
* Non-Repudiation: *YES/NO*

---

## Exercise 9.2: Automatic Updates with a hawkBit Update Server

This section gives a step by step explanation how to distribute OTA updates with a hawkBit update server.

### Start hawkBit Docker Container

Open a terminal in this virtual machine:

In [ ]:
execute_vm_command('gnome-terminal')

Start the hawkBit server based on a corresponding docker container by issuing the following line in the terminal:

``sudo docker run -p 8080:8080 hawkbit/hawkbit-update-server:0.3.0 --hawkbit.dmf.rabbitmq.enabled=false --hawkbit.server.ddi.security.authentication.anonymous.enabled=true``

Wait until the terminal shows a line similar to:

``: Started Start in XX.XXX seconds (JVM running for XX.XXX)``

### Open hawkBit Frontend

Open a new tab in Firefox and enter ``localhost:8080``.

Login with the credentials ``admin:admin``.
<br>
<img src="img/hawkbit-login.png" width="300">
<br>

### Set Polling Time

In the menu on the left, click *System Config* and scroll down to set the *Polling Time* that is communicated to connected devices by the hawkBit server to 1 minute. Afterward, click on the left symbol at the bottom to save the configuration.

<br>
<img src="img/hawkbit_set_polling_time.png" width="300">
<br>

### Configure and Start SWUpdate

Login to your board by SSH.

First, the post-update script that runs after a successful update has to be extended to report a successful update to the hawkBit server. This can be achieved by adding the following line to the file ``/etc/swupdate/postupdate.sh`` right before the ``reboot`` command:

```
timeout 5 swupdate -u '-t default -u http://192.168.13.17:8080 -i Testboard1 -c 2'
```

Then, start the ``swupdate`` daemon in ``suricatta`` mode (``-u``):

```bash
swupdate -u '-t default -u http://192.168.13.17:8080 -i Testboard1' -e 'stable,rootfs-a'
```

**Hint:**<br>
If your board's currently active root file system is ``rootfs-a``, then replace ``rootfs-a`` with ``rootfs-b`` in the command above.

### Connect the Board

Check if the *Targets* list in the *Deployment* menu already shows a device named *Testboard1*.

<br>
<img src="img/hawkbit_new_device.png" width="200">
<br>


<br>
<img src="img/hawkbit_notification.png" width="200">
<br>


Click on the notification alarm symbol in the top right corner to refresh the target list.

Then, a new device named *Testboard1* should show up within the *Targets* list in the *Deployment* menu

<br>
<img src="img/hawkbit_new_target.png" width="300">
<br>


### Create Update Distribution and Software Module

Navigate to the *Distributions* menu within the left menu bar.

Create a new distribution configuration by clicking the plus symbol *Create..*

<br>
<img src="img/hawkbit_create_new_distro.png" width="300">
<br>

Fill the mandatory fields, e.g. as shown here:

<br>
<img src="img/hawkbit_distro_naming.png" width="200">
<br>

Create a software module:
<br>
<img src="img/hawkbit_create_new_software.png" width="300">
<br>

Select type *OS* and set *Name* and *Version*:
<br>
<img src="img/hawkbit_software_naming.png" width="200">
<br>

Afterwards, it should look like this:

<br>
<img src="img/hawkbit_sw_distro.png" width="600">
<br>

Now, drag and drop the software module to the distribution to connect them.

<br>
<img src="img/hawkbit_sw_distro2.png" width="600">
<br>

Confirm this by pressing *OK*:
<br>
<img src="img/hawkbit_sw_distro3.png" width="400">
<br>

If you select the distribution, the connected software module should be highlighted in green.

<br>
<img src="img/hawkbit_sw_distro4.png" width="600">
<br>


### Upload File for Software Module

The next step is to upload an update file for the created software module.

Navigate to the *Upload* menu:
<br>
<img src="img/hawkbit_upload1.png" width="200">
<br>

Select the software module (left click) and open a file dialog by clicking *Upload File* on bottom right.
<br>
<img src="img/hawkbit_upload2.png" width="300">
<br>

Navigate to your ``.swu`` file and select it by pressing *OK*.

Uploading the file might take some time...

After a successful upload it should look similar to this:
<br>
<img src="img/hawkbit_upload3.png" width="900">
<br>



### Update Rollout

Navigate back to the *Deployment* menu.

The previously created distribution should now show up.
<br>
<img src="img/hawkbit_deploy1.png" width="500">
<br>

One can initiate the OTA update for the connected board by drag and drop of the distribution to the taget:

<br>
<img src="img/hawkbit_deploy2.png" width="500">
<br>

Confirm a forced update by pressing *OK*.

<br>
<img src="img/hawkbit_deploy3.png" width="500">
<br>

A pending update action should be added to the action history of your target board.

<br>
<img src="img/hawkbit_deploy4.png" width="900">
<br>

The board polls regulary (in a 1-minute interval, as configured above) for new updates and the update process might also take some time.

Therefore, the only thing you can do at this stage is ... waiting ... and discussing!

### Discussion

What are advantages and disadvantages of manual or automatic updates?

**Manual Update by Web Interface:**

Advantages:
* *TODO*
* *TODO*

Disadvantages:
* *TODO*
* *TODO*

**Automatic Update by hawkBit:**

Advantages:
* *TODO*
* *TODO*

Disadvantages:
* *TODO*
* *TODO*

A successful update is visualized by a green symbol.
<br>
<img src="img/hawkbit_deploy5.png" width="900">
<br>

### Discussion

Which error case might go unnoticed, even if the green symbol occurs?

And how could the implementation be improved to cover that error case?

**Error Case Not Covered:**

*TODO*

**Possible Improvement:**

*TODO*